In [1]:
import pandas as pd
import os

In [4]:
tomato = pd.read_csv("../data/cleaned/tomato_2023_2025_clean.csv")
potato = pd.read_csv("../data/cleaned/potato_2023_2025_clean.csv")
onion  = pd.read_csv("../data/cleaned/onion_2023_2025_clean.csv")
rice   = pd.read_csv("../data/cleaned/rice_2023_2025_clean.csv")
wheat  = pd.read_csv("../data/cleaned/wheat_2023_2025_clean.csv")

In [5]:
commodity = pd.concat(
    [
        tomato,
        potato,
        onion,
        rice,
        wheat
    ],
    ignore_index=True
)

commodity.shape

(5429, 8)

In [6]:
weather = pd.read_csv(
    "../data/external/weather_maharashtra_2023_2025.csv"
)

weather.head()

,Date,District,Max_Temp,Min_Temp,Rainfall_mm,Rain_mm,WindSpeed
0,2023-01-01,Ahmednagar,28.9,14.3,0.0,0.0,14.5
1,2023-01-02,Ahmednagar,28.7,14.4,0.0,0.0,11.4
2,2023-01-03,Ahmednagar,28.5,14.3,0.0,0.0,10.9
3,2023-01-04,Ahmednagar,28.6,15.2,0.0,0.0,16.6
4,2023-01-05,Ahmednagar,27.7,17.6,0.1,0.1,19.0


In [8]:
events = pd.read_csv(
    "../data/external/events_2023_2025.csv"
)

events.head()

,Date,Festival,Holiday,Month,Year,Day,Weekend
0,2023-01-26,Republic Day,1,1,2023,26,False
1,2023-03-08,Holi,1,3,2023,8,False
2,2023-03-22,Gudi Padwa,1,3,2023,22,False
3,2023-04-04,Mahavir Jayanti,1,4,2023,4,False
4,2023-04-07,Good Friday,1,4,2023,7,False


In [9]:
commodity["Date"] = pd.to_datetime(commodity["Date"])

weather["Date"] = pd.to_datetime(weather["Date"])

events["Date"] = pd.to_datetime(events["Date"])

In [10]:
df = commodity.merge(
    weather,
    on="Date",
    how="left"
)

df.shape

(54290, 14)

In [11]:
df = df.merge(
    events,
    on="Date",
    how="left"
)

df.shape

(54340, 20)

In [12]:
df["Festival"] = df["Festival"].fillna("No Festival")

df["Holiday"] = df["Holiday"].fillna(0)

df["Weekend"] = df["Weekend"].fillna(False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_13376\4251353185.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Weekend"] = df["Weekend"].fillna(False)


In [13]:
df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Day"] = df["Date"].dt.day

df["DayOfWeek"] = df["Date"].dt.dayofweek

df["Quarter"] = df["Date"].dt.quarter

In [14]:
df = df.sort_values(
    [
        "Commodity",
        "Date"
    ]
)

In [15]:
df["Lag_1"] = (
    df
    .groupby("Commodity")["Modal_Price"]
    .shift(1)
)

In [16]:
df["Lag_2"] = (
    df
    .groupby("Commodity")["Modal_Price"]
    .shift(2)
)

In [17]:
df["Lag_3"] = (
    df
    .groupby("Commodity")["Modal_Price"]
    .shift(3)
)

In [18]:
df["Rolling_7"] = (

    df
    .groupby("Commodity")["Modal_Price"]
    .rolling(7)
    .mean()
    .reset_index(level=0,drop=True)

)

In [20]:
df["Rolling_30"] = (

    df
    .groupby("Commodity")["Modal_Price"]
    .rolling(30)
    .mean()
    .reset_index(level=0,drop=True)

)

In [21]:
df["Price_Change"] = (

    df
    .groupby("Commodity")["Modal_Price"]
    .diff()

)

In [22]:
df = df.fillna(method="bfill")

df = df.fillna(method="ffill")

C:\Users\DELL\AppData\Local\Temp\ipykernel_13376\2480474418.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="bfill")
C:\Users\DELL\AppData\Local\Temp\ipykernel_13376\2480474418.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54340 entries, 21740 to 54339
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   State            54340 non-null  object        
 1   Commodity_Group  54340 non-null  object        
 2   Commodity        54340 non-null  object        
 3   Date             54340 non-null  datetime64[ns]
 4   Arrival          54340 non-null  float64       
 5   Arrival_Unit     54340 non-null  object        
 6   Modal_Price      54340 non-null  float64       
 7   Price_Unit       54340 non-null  object        
 8   District         54340 non-null  object        
 9   Max_Temp         54340 non-null  float64       
 10  Min_Temp         54340 non-null  float64       
 11  Rainfall_mm      54340 non-null  float64       
 12  Rain_mm          54340 non-null  float64       
 13  WindSpeed        54340 non-null  float64       
 14  Festival         54340 non-null  object

In [24]:
df.head()

,State,Commodity_Group,Commodity,Date,Arrival,Arrival_Unit,Modal_Price,Price_Unit,District,Max_Temp,...,Day,Weekend,DayOfWeek,Quarter,Lag_1,Lag_2,Lag_3,Rolling_7,Rolling_30,Price_Change
21740,Maharashtra,Vegetables,Onion,2023-01-01,4846.0,Metric Tonnes,1369.7,Rs./Quintal,Ahmednagar,28.9,...,1,False,6,1,1369.7,1369.7,1369.7,1369.7,1358.006667,0.0
21741,Maharashtra,Vegetables,Onion,2023-01-01,4846.0,Metric Tonnes,1369.7,Rs./Quintal,Pune,29.9,...,1,False,6,1,1369.7,1369.7,1369.7,1369.7,1358.006667,0.0
21742,Maharashtra,Vegetables,Onion,2023-01-01,4846.0,Metric Tonnes,1369.7,Rs./Quintal,Nashik,28.4,...,1,False,6,1,1369.7,1369.7,1369.7,1369.7,1358.006667,0.0
21743,Maharashtra,Vegetables,Onion,2023-01-01,4846.0,Metric Tonnes,1369.7,Rs./Quintal,Solapur,31.4,...,1,False,6,1,1369.7,1369.7,1369.7,1369.7,1358.006667,0.0
21744,Maharashtra,Vegetables,Onion,2023-01-01,4846.0,Metric Tonnes,1369.7,Rs./Quintal,Nagpur,27.4,...,1,False,6,1,1369.7,1369.7,1369.7,1369.7,1358.006667,0.0


In [25]:
os.makedirs(
    "../data/final",
    exist_ok=True
)

df.to_csv(

    "../data/final/final_dataset.csv",

    index=False

)

print("Saved Successfully")

Saved Successfully


In [26]:
print(df.shape)

print(df.isnull().sum())

print(df.duplicated().sum())

print(df.describe())

(54340, 28)
State              0
Commodity_Group    0
Commodity          0
Date               0
Arrival            0
Arrival_Unit       0
Modal_Price        0
Price_Unit         0
District           0
Max_Temp           0
Min_Temp           0
Rainfall_mm        0
Rain_mm            0
WindSpeed          0
Festival           0
Holiday            0
Month              0
Year               0
Day                0
Weekend            0
DayOfWeek          0
Quarter            0
Lag_1              0
Lag_2              0
Lag_3              0
Rolling_7          0
Rolling_30         0
Price_Change       0
dtype: int64
0
                                Date       Arrival   Modal_Price  \
count                          54340  54340.000000  54340.000000   
mean   2024-07-02 02:15:24.843577600   5378.557335   2552.735193   
min              2023-01-01 00:00:00      0.100000    349.390000   
25%              2023-09-29 00:00:00    439.940000   1346.360000   
50%              2024-07-03 12:00:00   1551.0

In [27]:
(df["Rainfall_mm"] == df["Rain_mm"]).all()

np.True_

In [28]:
df = df.drop(columns=["Rain_mm"])

In [29]:
df = df.drop(columns=["Arrival_Unit", "Price_Unit"])